# W7 作業｜建立並比較三個模型
**智慧醫療與機器學習基礎**　臺北醫學大學 智慧醫療學士學位學程　授課教師：王三源　課程助教：古珉瑄（醫資所，m610115009@tmu.edu.tw）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wang3yuan/FIHML/blob/main/homework/W07_HW.ipynb)

**開始前**：先「檔案 → 在雲端硬碟中儲存複本」，在自己的複本上作答。
**繳交**：完成後「檔案 → 下載 → 下載 .ipynb」，上傳至 Im@TMU。**期限：11/1（日）23:59**（W8 光復節補假，所以有兩週）
**檔名**：`W07_HW_學號_姓名.ipynb`

### 情境
你是記憶門診的資料分析助理。門診想用初診時的評估資料，先篩出「需要進一步做完整失智評估」的病人。請用課堂學過的三種模型建立預測，並建議門診該用哪一個、門檻定多少。

### 資料
延續 W5、W6 的 Kaggle [Alzheimer's Disease Dataset](https://www.kaggle.com/datasets/rabieelkharoua/alzheimers-disease-dataset)（Rabie El Kharoua, 2024，CC BY 4.0；**原資料為合成資料**），使用清理好的 `alzheimers_clean.csv`（2,149 位病人）。
`Diagnosis` 1 = 阿茲海默症（AD）、0 = 非 AD。主要欄位：`MMSE` 0–30、`FunctionalAssessment` 0–10、`ADL` 0–10（越低越差）、`MemoryComplaints`、`BehavioralProblems`（1 = 有）。

### 執行環境
本作業已在 **scikit-learn 1.6.1／pandas 2.2.3** 與 **scikit-learn 1.9.1／pandas 3.0.6** 測試通過。查手冊時請選擇和自己相同的版本（第一格會印出來）。

### 評分標準（100 分）
| 項目 | 配分 | 評分重點 |
|---|---|---|
| 1. 準備資料 | 10 | X、y 正確（`Ethnicity` 轉成 0／1）、切分正確、說明哪些資料不能放進 X |
| 2. 決策樹 | 20 | 五種深度的正確率、畫出樹、解讀第一刀、指出過度擬合 |
| 3. 隨機森林 | 15 | 正確率、前 10 名特徵重要性圖、和決策樹比較 |
| 4. 羅吉斯回歸 | 15 | 用 pipeline 標準化、係數解讀、勝算比的計算與解讀 |
| 5. 比較與門檻 | 25 | 五個指標的比較表、調整門檻、給門診的建議有理由 |
| 6. 查手冊 | 15 | 參數資訊正確（含版本）、實驗設計、結果說明 |
| GenAI 使用說明 | 必填 | 如實揭露不扣分；使用了卻未揭露，作業成績上限 60 分 |

> 💡 程式寫法可以參考課堂 notebook（`notebooks/W07_sklearn.ipynb`）。卡住時可以問 Colab 的 Gemini，但要對照手冊查證，並在最後的「GenAI 使用說明」揭露。
> 💡 繳交前「執行階段 → 重新啟動並全部執行」，確認從頭到尾沒有錯誤。

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import urllib.request
import matplotlib.font_manager as fm

print("Python", sys.version.split()[0], "| scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

# 中文字型（與課堂相同）
for name in ["NotoSansTC-Regular.ttf", "NotoSansTC-Bold.ttf"]:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/wang3yuan/FIHML/main/fonts/" + name, name)
    fm.fontManager.addfont(name)
plt.rcParams["font.family"] = "Noto Sans TC"
plt.rcParams["axes.unicode_minus"] = False

C_NO, C_YES = "#00897B", "#D06F1A"   # 非 AD、AD
ad = pd.read_csv("https://raw.githubusercontent.com/wang3yuan/FIHML/main/data/hw/alzheimers_clean.csv")
print(ad.shape)
ad.head()

---
## 1. 準備資料（10 分）
**(a)** 建立特徵 `X` 與標籤 `y`：
- `X`：拿掉 `PatientID`（編號不是特徵）與 `Diagnosis`（答案）
- `Ethnicity` 是**類別**（0–3 代表不同族群，數字沒有大小的意義），用 `pd.get_dummies(X, columns=["Ethnicity"], drop_first=True, dtype=int)` 轉成 0／1 欄位
- `y`：`Diagnosis`

**(b)** 用 `train_test_split` 切成訓練集與測試集（`test_size=0.25`、`stratify=y`、`random_state=1151`），印出兩份的大小與 AD 比例。

In [ ]:
from sklearn.model_selection import train_test_split

# (a) 建立 X（不含 PatientID、Diagnosis；Ethnicity 用 get_dummies）與 y
# TODO

# (b) 切分訓練集與測試集，印出兩份的大小與 AD 比例
# TODO


**(c)** 為什麼 `Diagnosis` 不能放進 `X`？如果資料裡還有一欄「是否已開始服用失智症藥物」，可以放進 `X` 嗎？為什麼？

> 你的回答：


---
## 2. 決策樹（20 分）
**(a)** 用 `max_depth` = 2、3、4、6、`None` 各建一棵決策樹（`random_state=1151`），印出訓練集與測試集的正確率。

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# TODO


**(b)** 建立 `max_depth=3` 的決策樹（`random_state=1151`），存成變數 `tree`，用 `plot_tree` 畫出來（`class_names=["非 AD", "AD"]`）。

In [ ]:
# TODO


**(c)** 回答：
1. 哪個深度的測試集正確率最高？不限深度（`None`）的結果說明了什麼？
2. `max_depth=3` 的樹，第一刀用了哪個特徵、切點是多少？用一句話解讀，臨床上合理嗎？

> 你的回答：


---
## 3. 隨機森林（15 分）
**(a)** 訓練 `RandomForestClassifier(random_state=1151)`，存成變數 `forest`，印出訓練集與測試集的正確率。

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# TODO


**(b)** 用 `ax.barh` 畫出前 10 名的特徵重要性（`feature_importances_`），最重要的在最上面。

In [ ]:
# TODO


**(c)** 前 3 名是哪些特徵？和第 2 題決策樹的第一刀一致嗎？

> 你的回答：


---
## 4. 羅吉斯回歸（15 分）
**(a)** 用 `make_pipeline(StandardScaler(), LogisticRegression())` 訓練，存成變數 `logit`，印出訓練集與測試集的正確率。

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# TODO


**(b)** 取出係數（`logit[-1].coef_[0]`），畫成長條圖或依大小排序列出。找出**最能增加**與**最能降低** AD 機率的特徵各 2 個。

**(c)** 從 (b) 選一個係數，用 `np.exp()` 算出勝算比，用一句話解讀（標準化之後，「增加 1 個單位」＝增加 1 個標準差）。

In [ ]:
# TODO


> 你的回答：


---
## 5. 比較與門檻（25 分）
**(a)** 用同一份測試集，比較第 2 題的 `tree`（`max_depth=3`）、第 3 題的 `forest`、第 4 題的 `logit`：整理成一張表，包含**正確率、敏感度、特異度、F1、AUC**（參考課堂 notebook 3.2 的程式）。

In [ ]:
# TODO


**(b)** 選一個你認為最適合的模型，把判斷為 AD 的門檻分別設成 0.5、0.4、0.3，計算敏感度與特異度。

In [ ]:
# TODO


**(c)** 給門診的建議：你會用哪個模型、門檻定多少？請從「漏掉一位 AD 病人的代價」和「誤報一位非 AD 病人的代價」說明理由。

**(d)** 課堂上（長照模擬資料）羅吉斯回歸的正確率最好；這份資料是哪個模型最好？為什麼不同的資料，最好的模型可能不一樣？

> 你的回答：


---
## 6. 查手冊：一個課堂沒調過的參數（15 分）
從 `DecisionTreeClassifier`、`RandomForestClassifier`、`LogisticRegression`、`KNeighborsClassifier`、`SVC` 選一個模型，在 [scikit-learn 手冊](https://scikit-learn.org/stable/)（**選和自己相同的版本**）找一個**課堂沒有調過**的參數（例如 `criterion`、`class_weight`、`max_features`、`C`、`weights`；課堂用過的 `max_depth`、`min_samples_leaf`、`n_estimators` 不算）。

**(a)** 填表：

| 項目 | 你的答案 |
|---|---|
| 模型 | |
| 參數名稱 | |
| 型別與預設值（照抄手冊） | |
| 用自己的話說明意思 | |
| 你查的 scikit-learn 版本 | |

**(b)** 實驗：比較預設值和你設定的值（至少兩個值），印出測試集的結果（正確率，或其他適合的指標）。

**(c)** 用一句話說明結果。

In [ ]:
# TODO


> 你的回答：


---
## GenAI 使用說明（必填，不計分）
依課程〈生成式 AI 使用指引〉：**如實揭露不會扣分**；使用了卻未揭露，作業成績上限 60 分。沒有使用也請寫「本次作業未使用 GenAI」。

> 本次作業是否使用 GenAI：
>
> 用了哪些工具（例如 Colab 的 Gemini、ChatGPT）：
>
> 用在哪一題、怎麼問（例如：第 4 題請 Gemini 用中文解釋勝算比）：
>
> 怎麼查證、有沒有修改它給的內容：